In [ ]:
# NOTE : lightcurves_DASCH.ipynb - Cell 1

"""
Batch-fetch DASCH light curves for a list of stars.

Reads star names from a plain-text list (one name per line, '#' starts a
comment, blank lines ignored -- same format as rcb_Bright_Targets.txt),
queries the DASCH DR7 API via the `daschlab` package for each star, and
writes one CSV per star plus a summary log.

Install requirements:
    pip install daschlab

Usage:
    python fetch_dasch_lightcurves.py
    python fetch_dasch_lightcurves.py --star-list path\\to\\list.txt --out-dir path\\to\\output
"""

import argparse
import csv
import sys
import time
import traceback
from pathlib import Path

from daschlab import Session
from daschlab.query import NoSolutionError  # raised when a name fails to resolve


DEFAULT_STAR_LIST = Path("rcb_Bright_Targets.txt")
DEFAULT_OUT_DIR = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\lightcurves_DASCH"
)


def parse_star_list(path: Path) -> list[str]:
    """Parse the star-list text format: one name per line, '#' = comment."""
    names = []
    with open(path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            names.append(line)
    return names


def safe_filename(name: str) -> str:
    """Turn a star name like 'V854 Cen' into a filesystem-safe stem."""
    keep = "".join(c if c.isalnum() or c in " _-+" else "_" for c in name)
    return keep.strip().replace(" ", "_")


def fetch_one(star_name: str, sessions_dir: Path, out_dir: Path) -> tuple[str, str]:
    """
    Fetch and save the light curve for a single star.
    Returns (status, detail) where status is 'ok', 'no_resolve', or 'error'.
    """
    stem = safe_filename(star_name)
    session_path = sessions_dir / stem
    csv_path = out_dir / f"{stem}_dasch_lightcurve.csv"

    try:
        sess = Session(session_path)
        sess.select_target(name=star_name)
        sess.select_refcat("apass")
        lc = sess.lightcurve(0)  # closest catalog source to the target
        lc.write(csv_path, format="ascii.csv", overwrite=True)
        return "ok", f"{len(lc)} rows -> {csv_path.name}"
    except NoSolutionError as e:
        return "no_resolve", str(e)
    except Exception as e:  # noqa: BLE001 - want to log and keep going per-star
        return "error", f"{type(e).__name__}: {e}"


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--star-list", type=Path, default=DEFAULT_STAR_LIST,
                     help="Path to the star list text file")
    ap.add_argument("--out-dir", type=Path, default=DEFAULT_OUT_DIR,
                     help="Directory to write light curve CSVs and the summary log")
    ap.add_argument("--delay", type=float, default=1.0,
                     help="Seconds to pause between API queries (be polite to the server)")
    args = ap.parse_args()

    out_dir = args.out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    sessions_dir = out_dir / "_sessions"  # per-star daschlab working dirs (cache)
    sessions_dir.mkdir(exist_ok=True)

    if not args.star_list.exists():
        sys.exit(f"Star list not found: {args.star_list}")

    stars = parse_star_list(args.star_list)
    print(f"Found {len(stars)} stars in {args.star_list}")

    summary_path = out_dir / "fetch_summary.csv"
    results = []

    for i, star in enumerate(stars, 1):
        print(f"[{i}/{len(stars)}] {star} ... ", end="", flush=True)
        try:
            status, detail = fetch_one(star, sessions_dir, out_dir)
        except Exception:
            status, detail = "error", traceback.format_exc(limit=1)
        print(f"{status}: {detail}")
        results.append({"star": star, "status": status, "detail": detail})
        time.sleep(args.delay)

    with open(summary_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["star", "status", "detail"])
        writer.writeheader()
        writer.writerows(results)

    ok = sum(1 for r in results if r["status"] == "ok")
    failed = [r["star"] for r in results if r["status"] != "ok"]
    print(f"\nDone: {ok}/{len(stars)} light curves saved to {out_dir}")
    if failed:
        print(f"Failed to fetch: {', '.join(failed)}")
    print(f"Summary log: {summary_path}")


if __name__ == "__main__":
    main()